# Customer Segmentation - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on customer data to understand patterns, relationships, and prepare for clustering analysis.

## Objectives
- Load and validate customer data
- Understand data distribution and quality
- Identify patterns and correlations
- Generate insights for feature engineering
- Prepare data for clustering models

In [ ]:
# Import necessary libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from data.loader import DataLoader
from data.validator import DataValidator
from data.preprocessor import DataPreprocessor
from config.settings import settings
from config.logging_config import setup_logging, get_logger

# Set up logging
setup_logging()
logger = get_logger(__name__)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Data Loading and Initial Inspection

In [ ]:
# Initialize data loader
loader = DataLoader()

# Try to load existing data, otherwise generate sample data
try:
    # Look for CSV files in the raw data directory
    import os
    raw_data_files = [f for f in os.listdir('../data/raw') if f.endswith('.csv')]
    
    if raw_data_files:
        data_file = f'../data/raw/{raw_data_files[0]}'
        logger.info(f"Loading data from {data_file}")
        df = loader.load_data(data_file)
    else:
        logger.info("No data files found, generating sample data")
        df = loader.load_sample_data()
        # Save sample data for future use
        loader.save_data(df, '../data/raw/sample_customer_data.csv')
        
except Exception as e:
    logger.error(f"Error loading data: {e}")
    logger.info("Generating sample data instead")
    df = loader.load_sample_data()
    loader.save_data(df, '../data/raw/sample_customer_data.csv')

print(f"Data loaded successfully! Shape: {df.shape}")
print("\nFirst few rows:")
df.head()

In [ ]:
# Basic data information
print("=== Data Information ===")
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n=== Column Details ===")
print(df.info())

print("\n=== Statistical Summary ===")
display(df.describe())

## 2. Data Validation

In [ ]:
# Initialize validator
validator = DataValidator()

# Perform comprehensive validation
validation_result = validator.validate_dataframe(df)

print("=== Validation Results ===")
print(f"Is Valid: {validation_result.is_valid}")
print(f"Errors: {len(validation_result.errors)}")
print(f"Warnings: {len(validation_result.warnings)}")

if validation_result.errors:
    print("\n=== Errors ===")
    for error in validation_result.errors:
        print(f"❌ {error}")

if validation_result.warnings:
    print("\n=== Warnings ===")
    for warning in validation_result.warnings:
        print(f"⚠️ {warning}")

print("\n=== Summary ===")
for key, value in validation_result.summary.items():
    print(f"{key}: {value}")

In [ ]:
# Validate customer IDs specifically
id_validation = validator.validate_customer_id_uniqueness(df)

print("=== Customer ID Validation ===")
print(f"Valid: {id_validation.is_valid}")
for key, value in id_validation.summary.items():
    print(f"{key}: {value}")

# Get cleaning suggestions
suggestions = validator.suggest_data_cleaning(df)
print("\n=== Cleaning Suggestions ===")
for category, items in suggestions.items():
    if items:
        print(f"\n{category.upper()}:")
        for item in items:
            print(f"  • {item}")

## 3. Data Quality Analysis

In [ ]:
# Missing values analysis
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print("=== Missing Values Analysis ===")
display(missing_df[missing_df['Missing Count'] > 0])

# Visualize missing values
if missing_df['Missing Count'].sum() > 0:
    plt.figure(figsize=(12, 6))
    missing_df[missing_df['Missing Count'] > 0]['Missing Percentage'].plot(kind='bar')
    plt.title('Missing Values Percentage by Column')
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found!")

In [ ]:
# Duplicate rows analysis
duplicate_count = df.duplicated().sum()
print(f"=== Duplicate Rows Analysis ===")
print(f"Total duplicate rows: {duplicate_count}")
print(f"Duplicate percentage: {(duplicate_count / len(df)) * 100:.2f}%")

if duplicate_count > 0:
    print("\nFirst few duplicate rows:")
    display(df[df.duplicated(keep=False)].head(10))

## 4. Univariate Analysis

In [ ]:
# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Numeric columns distribution
if numeric_cols:
    # Create subplots for numeric columns
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    if n_rows == 1:
        axes = [axes] if n_cols == 1 else axes
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        if i < len(axes):
            axes[i].hist(df[col].dropna(), bins=30, alpha=0.7, edgecolor='black')
            axes[i].set_title(f'Distribution of {col}')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)
    
    # Hide unused subplots
    for i in range(len(numeric_cols), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Box plots for outlier detection
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    if n_rows == 1:
        axes = [axes] if n_cols == 1 else axes
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        if i < len(axes):
            axes[i].boxplot(df[col].dropna())
            axes[i].set_title(f'Box Plot of {col}')
            axes[i].set_ylabel(col)
            axes[i].grid(True, alpha=0.3)
    
    # Hide unused subplots
    for i in range(len(numeric_cols), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical columns analysis
if categorical_cols:
    for col in categorical_cols:
        print(f"\n=== {col.upper()} Analysis ===")
        value_counts = df[col].value_counts()
        print(f"Unique values: {df[col].nunique()}")
        print(f"Value counts:")
        print(value_counts)
        
        # Plot categorical distribution
        plt.figure(figsize=(10, 6))
        if df[col].nunique() <= 10:
            value_counts.plot(kind='bar')
            plt.title(f'Distribution of {col}')
            plt.ylabel('Count')
            plt.xticks(rotation=45)
        else:
            # For columns with many unique values, show top 10
            value_counts.head(10).plot(kind='bar')
            plt.title(f'Top 10 Values in {col}')
            plt.ylabel('Count')
            plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()

## 5. Bivariate Analysis

In [ ]:
# Correlation matrix for numeric columns
if len(numeric_cols) > 1:
    correlation_matrix = df[numeric_cols].corr()
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, fmt='.2f', cbar_kws={'label': 'Correlation'})
    plt.title('Correlation Matrix of Numeric Features')
    plt.tight_layout()
    plt.show()
    
    # Identify highly correlated pairs
    high_corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_value = correlation_matrix.iloc[i, j]
            if abs(corr_value) > 0.7:  # Threshold for high correlation
                high_corr_pairs.append({
                    'feature1': correlation_matrix.columns[i],
                    'feature2': correlation_matrix.columns[j],
                    'correlation': corr_value
                })
    
    if high_corr_pairs:
        print("\n=== Highly Correlated Feature Pairs (|r| > 0.7) ===")
        for pair in high_corr_pairs:
            print(f"{pair['feature1']} ↔ {pair['feature2']}: {pair['correlation']:.3f}")
    else:
        print("\nNo highly correlated feature pairs found (|r| > 0.7)")

In [ ]:
# Scatter plots for important numeric relationships
if len(numeric_cols) >= 2:
    # Select most important numeric features (excluding customer_id if present)
    important_numeric = [col for col in numeric_cols if col != 'customer_id']
    
    if len(important_numeric) >= 2:
        # Create pair plot for key numeric features
        key_features = important_numeric[:4]  # Limit to 4 features for readability
        
        if len(key_features) >= 2:
            plt.figure(figsize=(15, 10))
            sns.pairplot(df[key_features].dropna(), diag_kind='hist', plot_kws={'alpha': 0.6})
            plt.suptitle('Pair Plot of Key Numeric Features', y=1.02)
            plt.show()
            
        # Specific scatter plots with regression lines
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.flatten()
        
        plot_pairs = []
        for i in range(min(4, len(key_features)-1)):
            plot_pairs.append((key_features[i], key_features[i+1]))
        
        for i, (x_col, y_col) in enumerate(plot_pairs):
            if i < len(axes):
                axes[i].scatter(df[x_col], df[y_col], alpha=0.6, edgecolor='black', linewidth=0.5)
                
                # Add trend line
                z = np.polyfit(df[x_col].dropna(), df[y_col].dropna(), 1)
                p = np.poly1d(z)
                axes[i].plot(df[x_col], p(df[x_col]), "r--", alpha=0.8)
                
                axes[i].set_xlabel(x_col)
                axes[i].set_ylabel(y_col)
                axes[i].set_title(f'{x_col} vs {y_col}')
                axes[i].grid(True, alpha=0.3)
        
        # Hide unused subplots
        for i in range(len(plot_pairs), len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Categorical vs Numeric analysis
if categorical_cols and numeric_cols:
    # Excluding customer_id from numeric analysis
    analysis_numeric_cols = [col for col in numeric_cols if col != 'customer_id']
    
    for cat_col in categorical_cols[:2]:  # Limit to first 2 categorical columns
        if df[cat_col].nunique() <= 10:  # Only analyze columns with reasonable number of categories
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            axes = axes.flatten()
            
            for i, num_col in enumerate(analysis_numeric_cols[:4]):
                if i < len(axes):
                    # Box plot
                    df.boxplot(column=num_col, by=cat_col, ax=axes[i])
                    axes[i].set_title(f'{num_col} by {cat_col}')
                    axes[i].set_xlabel(cat_col)
                    axes[i].set_ylabel(num_col)
                    
            plt.suptitle(f'Numeric Features by {cat_col}', y=1.02)
            plt.tight_layout()
            plt.show()

## 6. Advanced Analysis

In [ ]:
# Customer segmentation potential analysis
print("=== Customer Segmentation Potential Analysis ===")

# Key metrics for segmentation
segmentation_features = []
if 'annual_income' in df.columns:
    segmentation_features.append('annual_income')
if 'spending_score' in df.columns:
    segmentation_features.append('spending_score')
if 'age' in df.columns:
    segmentation_features.append('age')
if 'purchase_frequency' in df.columns:
    segmentation_features.append('purchase_frequency')
if 'avg_transaction_value' in df.columns:
    segmentation_features.append('avg_transaction_value')

print(f"Key segmentation features identified: {segmentation_features}")

if len(segmentation_features) >= 2:
    # Create segmentation potential scatter plot
    fig = px.scatter_matrix(
        df[segmentation_features],
        title="Customer Segmentation Feature Relationships",
        opacity=0.6,
        height=800
    )
    fig.show()
    
    # RFM-like analysis if we have the data
    rfm_features = []
    if 'total_purchases' in df.columns:
        rfm_features.append('total_purchases')  # Frequency
    if 'avg_transaction_value' in df.columns:
        rfm_features.append('avg_transaction_value')  # Monetary
    if 'last_purchase_days_ago' in df.columns:
        rfm_features.append('last_purchase_days_ago')  # Recency (inverse)
    
    if len(rfm_features) >= 2:
        print(f"\nRFM Analysis features available: {rfm_features}")
        
        # Create RFM scatter plot
        if len(rfm_features) >= 2:
            plt.figure(figsize=(12, 8))
            
            # Create subplots for different RFM combinations
            plt.subplot(2, 2, 1)
            if len(rfm_features) >= 2:
                plt.scatter(df[rfm_features[0]], df[rfm_features[1]], alpha=0.6)
                plt.xlabel(rfm_features[0])
                plt.ylabel(rfm_features[1])
                plt.title(f'{rfm_features[0]} vs {rfm_features[1]}')
                plt.grid(True, alpha=0.3)
            
            if len(rfm_features) >= 3:
                plt.subplot(2, 2, 2)
                scatter = plt.scatter(df[rfm_features[0]], df[rfm_features[2]], 
                                    c=df[rfm_features[1]], alpha=0.6, cmap='viridis')
                plt.xlabel(rfm_features[0])
                plt.ylabel(rfm_features[2])
                plt.title(f'{rfm_features[0]} vs {rfm_features[2]} (colored by {rfm_features[1]})')
                plt.colorbar(scatter, label=rfm_features[1])
                plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()

In [ ]:
# Customer value analysis
print("=== Customer Value Analysis ===")

# Calculate customer lifetime metrics if possible
customer_metrics = {}

if 'annual_income' in df.columns and 'spending_score' in df.columns:
    # Create a simple customer value score
    df['customer_value_score'] = (df['annual_income'] / 1000) * (df['spending_score'] / 100)
    customer_metrics['customer_value_score'] = df['customer_value_score']
    
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.hist(df['customer_value_score'], bins=30, alpha=0.7, edgecolor='black')
    plt.title('Customer Value Score Distribution')
    plt.xlabel('Customer Value Score')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 2)
    plt.scatter(df['annual_income'], df['spending_score'], 
                c=df['customer_value_score'], alpha=0.6, cmap='viridis')
    plt.xlabel('Annual Income')
    plt.ylabel('Spending Score')
    plt.title('Income vs Spending Score')
    plt.colorbar(label='Customer Value Score')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 3)
    value_segments = pd.qcut(df['customer_value_score'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
    value_segments.value_counts().plot(kind='bar')
    plt.title('Customer Value Segments')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Customer Value Score Statistics:")
    print(df['customer_value_score'].describe())

## 7. Data Preprocessing Preview

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Show preprocessing steps that will be applied
print("=== Preprocessing Pipeline Preview ===")

# Identify column types
numeric_cols, categorical_cols = preprocessor.identify_column_types(df)
print(f"Numeric columns for processing: {numeric_cols}")
print(f"Categorical columns for processing: {categorical_cols}")

# Show preprocessing summary
print("\n=== Preprocessing Steps to be Applied ===")
print("1. Handle missing values (median for numeric, mode for categorical)")
print("2. Handle outliers using IQR method")
print("3. Encode categorical features (One-Hot Encoding)")
print("4. Scale numeric features (StandardScaler)")

In [ ]:
# Apply preprocessing for demonstration
print("=== Applying Preprocessing Pipeline ===")

df_processed = preprocessor.fit_transform(
    df,
    handle_missing=True,
    handle_outliers=True,
    encode_categorical=True,
    scale_features=True,
    encoding_method='onehot'
)

print(f"Original shape: {df.shape}")
print(f"Processed shape: {df_processed.shape}")
print(f"\nProcessed columns: {list(df_processed.columns)}")

# Show preprocessing summary
summary = preprocessor.get_preprocessing_summary()
print("\n=== Preprocessing Summary ===")
for key, value in summary.items():
    print(f"{key}: {value}")

# Display first few rows of processed data
print("\n=== Processed Data Sample ===")
display(df_processed.head())

## 8. Key Insights and Recommendations

In [ ]:
# Generate key insights from the EDA
print("=== KEY INSIGHTS FROM EDA ===")
print()

insights = []

# Data quality insights
if validation_result.is_valid:
    insights.append("✅ Data quality is good with no critical errors")
else:
    insights.append(f"⚠️ Data has {len(validation_result.errors)} errors that need attention")

if validation_result.warnings:
    insights.append(f"⚠️ {len(validation_result.warnings)} warnings identified for improvement")

# Feature insights
if len(numeric_cols) >= 3:
    insights.append(f"✅ Good variety of {len(numeric_cols)} numeric features for clustering")
else:
    insights.append(f"⚠️ Limited numeric features ({len(numeric_cols)}) may affect clustering quality")

if len(categorical_cols) > 0:
    insights.append(f"✅ {len(categorical_cols)} categorical features available for encoding")
else:
    insights.append("ℹ️ No categorical features to encode")

# Segmentation potential
if len(segmentation_features) >= 3:
    insights.append(f"✅ Strong segmentation potential with {len(segmentation_features)} key features")
elif len(segmentation_features) >= 2:
    insights.append(f"✅ Good segmentation potential with {len(segmentation_features)} key features")
else:
    insights.append("⚠️ Limited features available for segmentation")

# Correlation insights
if 'high_corr_pairs' in locals() and high_corr_pairs:
    insights.append(f"⚠️ {len(high_corr_pairs)} highly correlated feature pairs found - consider dimensionality reduction")
else:
    insights.append("✅ No concerning correlations found between features")

# Print insights
for insight in insights:
    print(insight)

print("\n=== RECOMMENDATIONS FOR CLUSTERING ===")
print()
recommendations = []

# Preprocessing recommendations
recommendations.append("1. Apply StandardScaler to normalize numeric features")
recommendations.append("2. Use One-Hot Encoding for categorical variables")
recommendations.append("3. Handle outliers using IQR method to prevent cluster distortion")

# Dimensionality reduction recommendations
if len(df_processed.columns) > 10:
    recommendations.append("4. Apply PCA to reduce dimensionality while preserving variance")
    recommendations.append("5. Use explained variance ratio to determine optimal number of components")

# Clustering recommendations
recommendations.append("6. Start with K-Means clustering (3-8 clusters)")
recommendations.append("7. Use Elbow method and Silhouette score for optimal cluster selection")
recommendations.append("8. Compare with Hierarchical clustering for validation")

# Evaluation recommendations
recommendations.append("9. Evaluate clusters using multiple metrics (Silhouette, Davies-Bouldin)")
recommendations.append("10. Generate business insights for each identified cluster")

for rec in recommendations:
    print(rec)

In [ ]:
# Save processed data for next phases
processed_data_path = '../data/processed/customer_data_processed.csv'
loader.save_data(df_processed, processed_data_path)
print(f"Processed data saved to: {processed_data_path}")

# Save preprocessor for future use
preprocessor_path = '../models/production/preprocessor.pkl'
preprocessor.save_preprocessor(preprocessor_path)
print(f"Preprocessor saved to: {preprocessor_path}")

# Save EDA insights
eda_summary = {
    'data_shape': df.shape,
    'processed_shape': df_processed.shape,
    'numeric_columns': numeric_cols,
    'categorical_columns': categorical_cols,
    'segmentation_features': segmentation_features,
    'validation_errors': len(validation_result.errors),
    'validation_warnings': len(validation_result.warnings),
    'high_correlations': len(high_corr_pairs) if 'high_corr_pairs' in locals() else 0,
    'insights': insights,
    'recommendations': recommendations
}

import json
with open('../reports/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)
print("EDA summary saved to: ../reports/eda_summary.json")

print("\n=== EDA COMPLETED SUCCESSFULLY ===")
print("Ready to proceed with Feature Engineering and PCA!")